# Network Security Monitoring using ML-based Anomaly Detection

5G and Beyond-5G (B5G/6G) networks enable high-bandwidth and
low-latency consumer applications such as video streaming,
augmented reality, and IoT services. However, these networks
are increasingly vulnerable to Distributed Denial of Service (DDoS)
attacks initiated by malicious User Equipments (UEs).

Traditional rule-based intrusion detection systems are not
sufficient to handle the scale, heterogeneity, and real-time
requirements of 5G networks. As a result, AI/ML-based
anomaly detection techniques are explored for proactive
network security monitoring.


## Base Paper

This project is based on the IEEE Transactions on Consumer Electronics
paper:

"Advancing Predictive Security for Consumer Applications in Beyond 5G/6G
Networks With Annotated Datasets" (Xylouris et al., 2025).

The paper studies DDoS attack detection in 5G networks using
UE-level network metrics collected from a real testbed and
evaluates multiple ML/DL models under a Network Data Analytics
Function (NWDAF) framework.


## Problem Statement

Given UE-level network performance and signaling metrics
collected from a real 5G testbed, the objective is to:

- Identify abnormal UE behavior corresponding to DDoS attacks
- Frame the task as a binary classification problem:
    - Benign UE traffic
    - Malicious (DDoS) UE traffic
- Evaluate ML/DL models using Precision, Recall, and F1-score

At this stage, the focus is on understanding the dataset
structure and feature composition.


## Dataset Description

The dataset used in this project is the NCSRD-DS-5GDDos v2.0 dataset.
It was recorded in a real-world 5G testbed aligned with 3GPP
specifications.

The dataset consists of multiple CSV files, including:
- amari_ue_data_*.csv : UE-level metrics
- enb_counters_*.csv  : Cell-level metrics
- mme_counters.csv   : Core network signaling metrics

In this project, we initially focus on the amari_ue_data file,
which contains UE identification, bearer information, cell
performance indicators, and uplink/downlink traffic statistics.

The dataset does not explicitly contain labels in its raw form.
Attack labels are derived later using known attack time windows
as described in the reference paper.


In [4]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

DATA_PATH = "C:/Users/saura/cn-project-network-security/data/amari_ue_data_classic_tabular.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)


In [5]:
df.shape


(424660, 80)

In [6]:
df.columns


Index(['_time', 'imeisv', '5g_tmsi', 'amf_ue_id', 'bearer_0_apn',
       'bearer_0_dl_total_bytes', 'bearer_0_ip', 'bearer_0_ipv6',
       'bearer_0_pdu_session_id', 'bearer_0_qos_flow_id', 'bearer_0_sst',
       'bearer_0_ul_total_bytes', 'bearer_1_apn', 'bearer_1_dl_total_bytes',
       'bearer_1_ip', 'bearer_1_pdu_session_id', 'bearer_1_qos_flow_id',
       'bearer_1_sst', 'bearer_1_ul_total_bytes', 'cell_1_cell_id',
       'cell_1_cqi', 'cell_1_dl_bitrate', 'cell_1_dl_err', 'cell_1_dl_mcs',
       'cell_1_dl_retx', 'cell_1_dl_tx', 'cell_1_epre', 'cell_1_initial_ta',
       'cell_1_p_ue', 'cell_1_pusch_snr', 'cell_1_ri',
       'cell_1_turbo_decoder_avg', 'cell_1_turbo_decoder_max',
       'cell_1_turbo_decoder_min', 'cell_1_ul_bitrate', 'cell_1_ul_err',
       'cell_1_ul_mcs', 'cell_1_ul_n_layer', 'cell_1_ul_path_loss',
       'cell_1_ul_phr', 'cell_1_ul_rank', 'cell_1_ul_retx', 'cell_1_ul_tx',
       'dl_bitrate', 'ran_id', 'ran_plmn', 'ran_ue_id', 'registered', 'rnti',
       't3

In [7]:
df.head()


,_time,imeisv,5g_tmsi,amf_ue_id,bearer_0_apn,bearer_0_dl_total_bytes,bearer_0_ip,bearer_0_ipv6,bearer_0_pdu_session_id,bearer_0_qos_flow_id,bearer_0_sst,bearer_0_ul_total_bytes,bearer_1_apn,bearer_1_dl_total_bytes,bearer_1_ip,bearer_1_pdu_session_id,bearer_1_qos_flow_id,bearer_1_sst,bearer_1_ul_total_bytes,cell_1_cell_id,cell_1_cqi,cell_1_dl_bitrate,cell_1_dl_err,cell_1_dl_mcs,cell_1_dl_retx,cell_1_dl_tx,cell_1_epre,cell_1_initial_ta,cell_1_p_ue,cell_1_pusch_snr,cell_1_ri,cell_1_turbo_decoder_avg,cell_1_turbo_decoder_max,cell_1_turbo_decoder_min,cell_1_ul_bitrate,cell_1_ul_err,cell_1_ul_mcs,cell_1_ul_n_layer,cell_1_ul_path_loss,cell_1_ul_phr,cell_1_ul_rank,cell_1_ul_retx,cell_1_ul_tx,dl_bitrate,ran_id,ran_plmn,ran_ue_id,registered,rnti,t3512,tac,tac_plmn,ue_aggregate_max_bitrate_dl,ue_aggregate_max_bitrate_ul,ul_bitrate,cell_3_cell_id,cell_3_cqi,cell_3_dl_bitrate,cell_3_dl_err,cell_3_dl_mcs,cell_3_dl_retx,cell_3_dl_tx,cell_3_epre,cell_3_initial_ta,cell_3_pusch_snr,cell_3_ri,cell_3_turbo_decoder_avg,cell_3_turbo_decoder_max,cell_3_turbo_decoder_min,cell_3_ul_bitrate,cell_3_ul_err,cell_3_ul_rank,cell_3_ul_retx,cell_3_ul_tx,bearer_1_ipv6,cell_3_p_ue,cell_3_ul_mcs,cell_3_ul_n_layer,cell_3_ul_path_loss,cell_3_ul_phr
0,2024-08-17 12:00:01.700000+00:00,3557821101183501,379786680,106,ims,4832474,192.168.4.6,2001:468:3000:2::,1,1,1,681308,internet,4673102.0,10.20.10.10,2.0,1.0,1.0,4660701.0,1.0,13.0,5484.0,0.0,19.4,1.0,20.0,-88.7,5.0,-12.0,43.0,2.0,2.0,2.0,2.0,2257.0,0.0,27.0,1.0,64.1,28.0,1.0,0.0,10.0,5484,74565,101,7,True,18060,1800,100,101,5000000000,2000000000,2257,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-08-17 12:00:06.845000+00:00,3557821101183501,379786680,106,ims,4832474,192.168.4.6,2001:468:3000:2::,1,1,1,681308,internet,4673522.0,10.20.10.10,2.0,1.0,1.0,4661121.0,1.0,13.0,5421.0,0.0,19.4,2.0,20.0,-89.8,5.0,-12.0,40.1,2.0,2.0,2.0,2.0,2252.0,0.0,27.0,1.0,64.2,28.0,1.0,0.0,10.0,5421,74565,101,7,True,18060,1800,100,101,5000000000,2000000000,2252,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-08-17 12:00:11.997000+00:00,3557821101183501,379786680,106,ims,4832474,192.168.4.6,2001:468:3000:2::,1,1,1,681308,internet,4674026.0,10.20.10.10,2.0,1.0,1.0,4661625.0,1.0,13.0,6272.0,0.0,19.5,2.0,23.0,-89.4,5.0,-12.0,39.8,2.0,2.0,2.0,2.0,2705.0,0.0,27.0,1.0,64.2,28.0,1.0,0.0,12.0,6272,74565,101,7,True,18060,1800,100,101,5000000000,2000000000,2705,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-08-17 12:00:17.156000+00:00,3557821101183501,379786680,106,ims,4832474,192.168.4.6,2001:468:3000:2::,1,1,1,681308,internet,4674446.0,10.20.10.10,2.0,1.0,1.0,4662045.0,1.0,13.0,5194.0,0.0,19.4,4.0,20.0,-89.8,5.0,-12.0,36.9,2.0,2.0,2.0,2.0,2245.0,0.0,27.0,1.0,64.4,28.0,1.0,0.0,10.0,5194,74565,101,7,True,18060,1800,100,101,5000000000,2000000000,2245,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-08-17 12:00:22.299000+00:00,3557821101183501,379786680,106,ims,4832474,192.168.4.6,2001:468:3000:2::,1,1,1,681308,internet,4674866.0,10.20.10.10,2.0,1.0,1.0,4662465.0,1.0,13.0,5401.0,0.0,19.4,2.0,20.0,-90.1,5.0,-12.0,44.3,2.0,2.0,2.0,2.0,2256.0,0.0,27.0,1.0,63.5,28.0,1.0,0.0,10.0,5401,74565,101,7,True,18060,1800,100,101,5000000000,2000000000,2256,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Number of unique UEs
df["imeisv"].nunique()


7

In [9]:
# Samples per UE
df["imeisv"].value_counts()


imeisv
8609960480859058    76275
8609960468879057    76266
8642840401594200    76205
3557821101183501    75724
8642840401624200    75304
8677660403123800    44885
8609960480666910        1
Name: count, dtype: int64

Each IMEISV corresponds to a specific UE device.
Certain UEs are known to initiate DDoS attacks,
while others generate benign traffic such as video streaming.


## Feature Categories

The dataset contains multiple categories of features:

1. UE Identification:
   - imeisv, rnti, amf_ue_id

2. Bearer-level Metrics:
   - bearer_*_ul_total_bytes
   - bearer_*_dl_total_bytes
   - bearer_*_pdu_session_id

3. Cell-level Radio Metrics:
   - cell_X_ul_bitrate, cell_X_dl_bitrate
   - cell_X_ul_retx, cell_X_dl_retx
   - cell_X_cqi, cell_X_mcs
   - cell_X_pusch_snr

4. Aggregated UE Traffic:
   - ul_bitrate, dl_bitrate
   - ue_aggregate_max_bitrate_ul/dl

These features are consistent with those used
in the reference paper for anomaly detection.


In [11]:
# Check timestamp column type
df["_time"].dtype


dtype('O')

In [13]:
# Convert timestamp to datetime (do NOT modify original yet)
df["_time"] = pd.to_datetime(df["_time"], errors="coerce")

df["_time"].head()


0   2024-08-17 12:00:01.700000+00:00
1   2024-08-17 12:00:06.845000+00:00
2   2024-08-17 12:00:11.997000+00:00
3   2024-08-17 12:00:17.156000+00:00
4   2024-08-17 12:00:22.299000+00:00
Name: _time, dtype: datetime64[ns, UTC]

In [14]:
# Time range of dataset
df["_time"].min(), df["_time"].max()


(Timestamp('2024-08-17 12:00:01.700000+0000', tz='UTC'),
 Timestamp('2024-08-22 06:59:55.404000+0000', tz='UTC'))

The dataset spans a continuous time interval during which
both benign and malicious traffic were generated. Time-based
information will later be used to derive attack labels based
on known attack windows, as described in the reference paper.

The dataset spans from August 17 to August 22, 2024 (UTC),
covering periods of both benign and malicious traffic.
This time range aligns with the attack scenarios described
in the reference paper, where multiple DDoS attack types
were launched during specific time windows.

The presence of accurate timestamp information enables
time-based labeling of attack and benign samples, which
will be performed in later stages of the project.


In [15]:
# Percentage of missing values per column
missing_percent = (df.isna().sum() / len(df)) * 100
missing_percent.sort_values(ascending=False).head(10)


bearer_1_ipv6               92.608204
cell_3_ul_n_layer           56.889512
cell_3_ul_mcs               56.889512
cell_3_turbo_decoder_avg    56.889276
cell_3_turbo_decoder_max    56.889276
cell_3_turbo_decoder_min    56.889276
cell_3_ul_phr               56.872321
cell_3_dl_mcs               56.872321
cell_3_ul_path_loss         56.872321
cell_3_p_ue                 56.872321
dtype: float64

In [16]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = df.select_dtypes(include=["object"]).columns

len(numeric_cols), len(categorical_cols)


(72, 6)

In [17]:
numeric_cols


Index(['imeisv', '5g_tmsi', 'amf_ue_id', 'bearer_0_dl_total_bytes',
       'bearer_0_pdu_session_id', 'bearer_0_qos_flow_id', 'bearer_0_sst',
       'bearer_0_ul_total_bytes', 'bearer_1_dl_total_bytes',
       'bearer_1_pdu_session_id', 'bearer_1_qos_flow_id', 'bearer_1_sst',
       'bearer_1_ul_total_bytes', 'cell_1_cell_id', 'cell_1_cqi',
       'cell_1_dl_bitrate', 'cell_1_dl_err', 'cell_1_dl_mcs', 'cell_1_dl_retx',
       'cell_1_dl_tx', 'cell_1_epre', 'cell_1_initial_ta', 'cell_1_p_ue',
       'cell_1_pusch_snr', 'cell_1_ri', 'cell_1_turbo_decoder_avg',
       'cell_1_turbo_decoder_max', 'cell_1_turbo_decoder_min',
       'cell_1_ul_bitrate', 'cell_1_ul_err', 'cell_1_ul_mcs',
       'cell_1_ul_n_layer', 'cell_1_ul_path_loss', 'cell_1_ul_phr',
       'cell_1_ul_rank', 'cell_1_ul_retx', 'cell_1_ul_tx', 'dl_bitrate',
       'ran_id', 'ran_plmn', 'ran_ue_id', 'rnti', 't3512', 'tac', 'tac_plmn',
       'ue_aggregate_max_bitrate_dl', 'ue_aggregate_max_bitrate_ul',
       'ul_bitrate', '

In [18]:
categorical_cols


Index(['bearer_0_apn', 'bearer_0_ip', 'bearer_0_ipv6', 'bearer_1_apn',
       'bearer_1_ip', 'bearer_1_ipv6'],
      dtype='object')

The dataset contains a large number of numerical features
related to radio metrics, throughput, and retransmissions,
which are suitable for ML-based anomaly detection.

Categorical features mainly include identifiers, IP addresses,
and configuration fields, which may be excluded or encoded
during preprocessing.


In [19]:
# Aggregate mean UL/DL bitrate per UE
ue_traffic = df.groupby("imeisv")[["ul_bitrate", "dl_bitrate"]].mean()

ue_traffic


,ul_bitrate,dl_bitrate
imeisv,,
3557821101183501,2.770605e+03,5.271866e+03
8609960468879057,1.202490e+07,2.560327e+06
8609960480666910,0.000000e+00,0.000000e+00
8609960480859058,6.043275e+06,2.535858e+06
8642840401594200,3.167807e+03,3.652884e+03
8642840401624200,2.044111e+05,4.932373e+03
8677660403123800,3.291392e+03,6.308657e+03


In [20]:
df[["ul_bitrate", "dl_bitrate"]].describe()


,ul_bitrate,dl_bitrate
count,4.246600e+05,4.246600e+05
mean,3.282706e+06,9.184303e+05
std,5.946027e+06,1.467131e+06
min,0.000000e+00,0.000000e+00
25%,2.479000e+03,3.247000e+03
50%,3.293000e+03,5.436000e+03
75%,5.815855e+06,3.111704e+06
max,2.011583e+07,3.652504e+06


## Summary 

In this notebook, we:
- Introduced the network security problem in 5G environments
- Aligned the project with the reference IEEE paper
- Understood the structure and semantics of the dataset
- Inspected UE-level, bearer-level, and cell-level features
- Verified timestamp coverage and feature completeness

No preprocessing, labeling, or modeling was performed.

The next notebook focuses on Exploratory Data Analysis (EDA)
to visualize and interpret traffic behavior.
